### DASHBOARD DE ACOMPANHAMENTO DE INVESTIMENTOS

Objetivo: automatizar a visualização de investimentos a partir de índices atualizados.

---

Ideias principais:

Criar um dashboard em PowerBI para visualização dos investimentos
- Porporção dos tipos de investimentos;
- Variação da cotação de cada investimento;
- DY;
- Histórico de variação dos preços;
- Renda fixa/Renda variável.

Back-end com API's Python.

---

Questões:
Como inserir novos dados? Tabela de compra/venda como input para o Python --> usa csv antigo e o novo, e cria um novo.

---

Adicional: implementar resumo dos investimentos com LLM.

---
### Podemos usar um SQL (SQLite ou PostgreSQL) para armazenar os dados de input e output. Talve
Input:
- ticker da ação;
- quantidade de cotas;
- valor comprado (ou quantidade de cotas);
- taxas de compra/venda;
- taxas de adm;

In [14]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


In [15]:
# import libraries
import pandas as pd
import numpy as np
import os
import requests
import warnings

from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime

In [16]:
# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Caminhos do projeto
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "historico_operacoes.csv"
OUTPUT_PATH = DATA_DIR / "carteira_atualizada.csv"

# Configurações do Pandas
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Ignorar warnings desnecessários
warnings.filterwarnings("ignore")

In [17]:
load_dotenv()

BASE_URL = "https://brapi.dev/api/v2"
TOKEN = os.getenv("BRAPI_TOKEN")

def get_stock_quote(symbols):
    url = f"{BASE_URL}/stocks/quote"

    params = {
        "symbols": ",".join(symbols),
        "token": TOKEN
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    # Pega o JSON da resposta
    json_response = response.json()
    resultados = json_response.get("results", [])

    # EXTRAÇÃO DO DICIONÁRIO 'data':
    # Percorre cada ação encontrada e extrai apenas a parte que importa (a chave 'data')
    dados_limpos = [acao['data'] for acao in resultados if 'data' in acao]
    
    return dados_limpos

# Chamando a função
dados_acoes = get_stock_quote(["PETR4", "VALE3", "ITUB4", "MGLU3"])

# Cria o DataFrame passando a lista de dicionários extraídos
df = pd.DataFrame(dados_acoes)

# visualização de algumas colunas
df_resumo = df[['shortName', 'regularMarketPrice', 'regularMarketDayHigh', 'regularMarketDayLow','regularMarketChangePercent','logourl']]
df_resumo

,shortName,regularMarketPrice,regularMarketDayHigh,regularMarketDayLow,regularMarketChangePercent,logourl
0,PETR4,42.47,42.76,41.72,0.90,https://icons.brapi.dev/icons/PETR4.svg
1,VALE3,71.41,71.80,70.69,0.15,https://icons.brapi.dev/icons/VALE3.svg
2,ITUB4,38.38,38.98,38.17,-1.59,https://icons.brapi.dev/icons/ITUB4.svg
3,MGLU3,4.23,4.25,3.90,8.18,https://icons.brapi.dev/icons/MGLU3.svg


### Principais dados a serem exportados da API
- Symbol (shortname) que representa o ticker da ação, por exemplo: ITUB4
- regularMarketPrice que mostra o preço atual da ação, permitindo calcular o valor atualizado da carteira
- regularMarketChangePercent indicando a variação percentual do dia. Podemos ainda adicionar o preço máximo e mínimo da ação no dia (high and low)
- logourl isso será importante no PowerBI, mostra o URL da logo da empresa.

Um dos pontos que a API Brapi não fornece é o dividendo anual (DY - dividend yield)

In [18]:
# preço médio para calcular 

def calcular_posicao_atual(df_ops):
    # Ordena por data para garantir que o cálculo cronológico esteja certo
    df_ops = df_ops.sort_values(by='Data')
    
    posicoes = {}
    
    for _, row in df_ops.iterrows():
        ticker = row['Ticker']
        op = row['Operacao'].upper()
        qtd = row['Quantidade']
        preco = row['Preco_Unitario']
        taxas = row['Taxas']
        
        # Se é a primeira vez que vemos o Ticker, cria o registro zerado
        if ticker not in posicoes:
            posicoes[ticker] = {'Qtd_Cotas': 0, 'Total_Investido': 0.0, 'Preco_Medio': 0.0}
            
        pos = posicoes[ticker]
        
        if op == 'COMPRA':
            # Custo total da compra (inclui taxas)
            custo_compra = (qtd * preco) + taxas
            
            # Atualiza totais
            pos['Qtd_Cotas'] += qtd
            pos['Total_Investido'] += custo_compra
            
            # Recalcula Preço Médio
            pos['Preco_Medio'] = pos['Total_Investido'] / pos['Qtd_Cotas']
            
        elif op == 'VENDA':
            # Venda altera a quantidade, mas o Preço Médio se mantém!
            pos['Qtd_Cotas'] -= qtd
            # Reduz o total investido proporcionalmente às cotas vendidas
            pos['Total_Investido'] -= (qtd * pos['Preco_Medio'])
            
            # Se vender tudo (zerar posição), zera tudo para evitar bugs
            if pos['Qtd_Cotas'] <= 0:
                pos['Qtd_Cotas'] = 0
                pos['Total_Investido'] = 0.0
                pos['Preco_Medio'] = 0.0
                
    # Transforma o dicionário final de volta em um DataFrame
    # E remove as ações que você zerou (vendeu tudo)
    df_carteira = pd.DataFrame.from_dict(posicoes, orient='index').reset_index()
    df_carteira = df_carteira.rename(columns={'index': 'Ticker'})
    df_carteira = df_carteira[df_carteira['Qtd_Cotas'] > 0]
    
    return df_carteira
    

In [19]:
try:
    df_operacoes = pd.read_csv(INPUT_PATH, sep=';') 
    print("Colunas encontradas no CSV:", df_operacoes.columns.tolist())
    
except FileNotFoundError:
    print(f"Erro: O arquivo {INPUT_PATH} não existe. Crie a planilha com o histórico primeiro!")
    exit()

# 3.2 Calcular Posição (Quantidade Atual e Preço Médio)
df_carteira = calcular_posicao_atual(df_operacoes)
print("\n--- Sua Carteira Calculada ---")
print(df_carteira)

# 3.3 Consultar a API Real
print("Consultando cotações atualizadas na Brapi...")
dados_api = get_stock_quote(df_carteira['Ticker'].tolist())
df_mercado = pd.DataFrame(dados_api)[['shortName', 'regularMarketPrice', 'regularMarketChangePercent','logourl']]

# 3.4 Lembre-se de ajustar o cruzamento para 'shortName'  Cruzar Carteira com Mercado
df_final = pd.merge(df_carteira, df_mercado, left_on='Ticker', right_on='shortName', how='left')

# 3.5 Calcular Métricas Finais
df_final['Valor_Atual'] = df_final['Qtd_Cotas'] * df_final['regularMarketPrice']
df_final['Lucro_Prejuizo_R$'] = df_final['Valor_Atual'] - df_final['Total_Investido']
df_final['Rentabilidade_%'] = (df_final['Lucro_Prejuizo_R$'] / df_final['Total_Investido']) * 100

# Limpar e Exportar
df_final = df_final.drop(columns=['shortName'])
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Output gerado com sucesso! Arquivo salvo em: {OUTPUT_PATH}")

df_final

Colunas encontradas no CSV: ['Data', 'Ticker', 'Operacao', 'Quantidade', 'Preco_Unitario', 'Taxas']

--- Sua Carteira Calculada ---
  Ticker  Qtd_Cotas  Total_Investido  Preco_Medio
0  ITUB4          4           150.00        37.50
1  MGLU3         10           500.00        50.00
2  VALE3          3            73.50        24.50
3  PETR4          5           192.50        38.50
Consultando cotações atualizadas na Brapi...

✅ Output gerado com sucesso! Arquivo salvo em: e:\linkedin\Projetos\investment_monitoring\data\carteira_atualizada.csv


,Ticker,Qtd_Cotas,Total_Investido,Preco_Medio,regularMarketPrice,regularMarketChangePercent,logourl,Valor_Atual,Lucro_Prejuizo_R$,Rentabilidade_%
0,ITUB4,4,150.00,37.50,38.38,-1.59,https://icons.brapi.dev/icons/ITUB4.svg,153.52,3.52,2.35
1,MGLU3,10,500.00,50.00,4.23,8.18,https://icons.brapi.dev/icons/MGLU3.svg,42.30,-457.70,-91.54
2,VALE3,3,73.50,24.50,71.41,0.15,https://icons.brapi.dev/icons/VALE3.svg,214.23,140.73,191.47
3,PETR4,5,192.50,38.50,42.47,0.90,https://icons.brapi.dev/icons/PETR4.svg,212.35,19.85,10.31


In [20]:
def registrar_nova_operacao(input_path):
    print("--- 📝 REGISTRO DE NOVA OPERAÇÃO ---")
    data = input("Data (AAAA-MM-DD) [Deixe em branco para hoje]: ")
    if not data:
        data = datetime.now().strftime("%Y-%m-%d")
        
    ticker = input("Ticker (ex: WEGE3): ").upper()
    operacao = input("Operação (Compra/Venda): ").capitalize()
    qtd = int(input("Quantidade: "))
    preco = float(input("Preço Unitário (R$): ").replace(',', '.'))
    taxas = float(input("Taxas totais (R$) [Deixe 0 se não houver]: ") or 0.0)
    
    nova_operacao = pd.DataFrame([{
        'Data': data,
        'Ticker': ticker,
        'Operacao': operacao,
        'Quantidade': qtd,
        'Preco_Unitario': preco,
        'Taxas': taxas
    }])
    
    # Se o arquivo já existe, anexa (append). Se não, cria um novo.
    try:
        df_existente = pd.read_csv(input_path, sep=';')
        df_atualizado = pd.concat([df_existente, nova_operacao], ignore_index=True)
    except FileNotFoundError:
        df_atualizado = nova_operacao
        
    df_atualizado.to_csv(input_path, sep=';', index=False)
    print(f"\n✅ Operação de {qtd} cotas de {ticker} salva com sucesso no histórico!")

In [21]:
# Para usar, basta chamar a função:
registrar_nova_operacao(INPUT_PATH)

--- 📝 REGISTRO DE NOVA OPERAÇÃO ---

✅ Operação de 5 cotas de ITUB4 salva com sucesso no histórico!
